# Model Error Analysis (per case_day)

This notebook analyzes per-`case_day` evaluation metrics (Dice) and joins them with:
- Label statistics (from `outputs/analysis/meta_info_table.csv`)
- Slice-level label occupancy (from `outputs/analysis/00_label_analysis/label_imbalance_per_slice.csv`)
- Intensity/style features + GMM clusters (from `outputs/analysis/case_day_intensity_features.csv` and `outputs/analysis/case_day_gmm_clusters.csv`)

All outputs are written under the model run directory:
- `outputs/20260206-053325_Unet3D/model_error_analysis/`


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "inputs" / "train.csv").exists():
            return p
        if (p / ".git").exists() and (p / "README.md").exists():
            return p
    raise RuntimeError(
        "Cannot find repo root. Start Jupyter from the project directory, or ensure inputs/train.csv exists."
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RUN_DIR = Path("outputs/20260206-053325_Unet3D")
EVAL_CSV = RUN_DIR / "train_eval" / "eval_per_case.csv"
OUT_DIR = RUN_DIR / "model_error_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

META_CSV = Path("outputs/analysis/meta_info_table.csv")
PER_SLICE_CSV = Path("outputs/analysis/00_label_analysis/label_imbalance_per_slice.csv")
INTENSITY_CSV = Path("outputs/analysis/case_day_intensity_features.csv")
CLUSTERS_CSV = Path("outputs/analysis/case_day_gmm_clusters.csv")

print("repo_root:", REPO_ROOT)
print("run_dir:", RUN_DIR)
print("eval_csv:", EVAL_CSV)
print("out_dir:", OUT_DIR)

for p in [EVAL_CSV, META_CSV, PER_SLICE_CSV, INTENSITY_CSV, CLUSTERS_CSV]:
    print(f"exists={p.exists()}  {p}")

assert EVAL_CSV.exists(), f"Not found: {EVAL_CSV}"
assert META_CSV.exists(), f"Not found: {META_CSV}"


## Load eval_per_case.csv

In [ ]:
eval_df = pd.read_csv(EVAL_CSV)
print("shape:", eval_df.shape)
print("columns:", eval_df.columns.tolist())
eval_df.head()

## Build label features per case_day

We aggregate `outputs/analysis/meta_info_table.csv` (row per `(slice, class)`), then pivot to one row per `case_day`.


In [ ]:
def build_case_day_label_features(meta_csv: Path) -> pd.DataFrame:
    usecols = [
        "case_day",
        "class",
        "label_area",
        "label_frac",
        "label_positive",
    ]
    meta = pd.read_csv(meta_csv, usecols=usecols)

    g = meta.groupby(["case_day", "class"], as_index=False)
    by = g.agg(
        n_slices=("label_positive", "size"),
        pos_slices=("label_positive", "sum"),
        pos_rate=("label_positive", "mean"),
        sum_area=("label_area", "sum"),
        mean_area=("label_area", "mean"),
        mean_frac=("label_frac", "mean"),
    )

    pos = meta.loc[meta["label_positive"]].copy()
    if len(pos) == 0:
        by["pos_mean_area"] = 0.0
        by["pos_mean_frac"] = 0.0
    else:
        gp = pos.groupby(["case_day", "class"], as_index=False).agg(
            pos_mean_area=("label_area", "mean"),
            pos_mean_frac=("label_frac", "mean"),
        )
        by = by.merge(gp, on=["case_day", "class"], how="left")
        by[["pos_mean_area", "pos_mean_frac"]] = by[["pos_mean_area", "pos_mean_frac"]].fillna(0.0)

    wide = by.pivot(index="case_day", columns="class")
    wide.columns = [f"label_{stat}_{cls}" for stat, cls in wide.columns]
    wide = wide.reset_index()
    return wide


label_wide = build_case_day_label_features(META_CSV)
label_wide.shape, label_wide.head()

## Load slice-level occupancy per case_day

This uses `outputs/analysis/00_label_analysis/label_imbalance_per_slice.csv` (one row per slice id) and aggregates to `case_day`.


In [ ]:
def build_case_day_slice_level_features(per_slice_csv: Path) -> pd.DataFrame:
    usecols = ["case_day", "any_positive", "total_area", "total_frac", "H", "W"]
    ps = pd.read_csv(per_slice_csv, usecols=usecols)

    def _mode_int(s: pd.Series) -> int:
        m = s.mode()
        if len(m):
            return int(m.iloc[0])
        return int(s.iloc[0])

    g = ps.groupby("case_day", as_index=False).agg(
        slices=("any_positive", "size"),
        any_pos_slices=("any_positive", "sum"),
        any_pos_rate=("any_positive", "mean"),
        total_area_sum=("total_area", "sum"),
        total_area_mean=("total_area", "mean"),
        total_frac_mean=("total_frac", "mean"),
        H_mode=("H", _mode_int),
        W_mode=("W", _mode_int),
    )
    g["any_pos_slices"] = g["any_pos_slices"].astype(int)
    return g


slice_feat = build_case_day_slice_level_features(PER_SLICE_CSV)
slice_feat.shape, slice_feat.head()

## Load intensity features and clustering assignments

In [ ]:
intensity_df = pd.read_csv(INTENSITY_CSV) if INTENSITY_CSV.exists() else None
clusters_df = pd.read_csv(CLUSTERS_CSV) if CLUSTERS_CSV.exists() else None

print("intensity_df:", None if intensity_df is None else intensity_df.shape)
print("clusters_df:", None if clusters_df is None else clusters_df.shape)

if intensity_df is not None:
    display(intensity_df[["case_day", "zero_ratio", "mean", "std", "p01", "p50", "p99", "entropy"]].head())
if clusters_df is not None:
    display(clusters_df.head())

## Join everything (one row per case_day)

Write the joined table to:
- `outputs/20260206-053325_Unet3D/model_error_analysis/eval_per_case_joined.csv`


In [ ]:
merged = eval_df.merge(label_wide, on="case_day", how="left")
merged = merged.merge(slice_feat, on="case_day", how="left")

if intensity_df is not None:
    merged = merged.merge(intensity_df, on="case_day", how="left")
if clusters_df is not None:
    merged = merged.merge(clusters_df, on="case_day", how="left")

print("merged shape:", merged.shape)
print("missing label_* rows:", int(merged.filter(regex=r"^label_").isna().any(axis=1).sum()))
if intensity_df is not None:
    print("missing intensity mean rows:", int(merged["mean"].isna().sum()))
if clusters_df is not None:
    print("missing cluster_id rows:", int(merged["cluster_id"].isna().sum()))

OUT_JOINED = OUT_DIR / "eval_per_case_joined.csv"
merged.to_csv(OUT_JOINED, index=False)
print("wrote:", OUT_JOINED)
merged.head()

## What does the model predict poorly?

In [ ]:
cols = ["case_day", "dice_mean", "dice_large_bowel", "dice_small_bowel", "dice_stomach", "num_slices"]
worst_mean = merged.sort_values("dice_mean", ascending=True).head(20)[cols]
worst_mean

In [ ]:
# If clustering is available, check cluster composition of the worst cases.
if "cluster_id" in merged.columns:
    display(worst_mean.merge(merged[["case_day", "cluster_id", "max_prob"]], on="case_day", how="left"))
    print("worst20 cluster_id counts:")
    display(worst_mean.merge(merged[["case_day", "cluster_id"]], on="case_day")[["cluster_id"]].value_counts().sort_index())


## Correlations (Pearson)

This is a quick diagnostic to see which metadata tracks overall performance.

In [ ]:
def corr_table(df: pd.DataFrame, target_cols: list[str], feature_cols: list[str]) -> pd.DataFrame:
    rows = []
    for t in target_cols:
        for f in feature_cols:
            a = df[t].to_numpy(dtype=float, copy=False)
            b = df[f].to_numpy(dtype=float, copy=False)
            mask = np.isfinite(a) & np.isfinite(b)
            if mask.sum() < 3:
                r = np.nan
            else:
                r = float(np.corrcoef(a[mask], b[mask])[0, 1])
            rows.append({"target": t, "feature": f, "pearson_r": r, "n": int(mask.sum())})
    out = pd.DataFrame.from_records(rows)
    out["abs_r"] = out["pearson_r"].abs()
    return out.sort_values(["target", "abs_r"], ascending=[True, False]).drop(columns=["abs_r"])


target_cols = [c for c in ["dice_mean", "dice_large_bowel", "dice_small_bowel", "dice_stomach"] if c in merged.columns]
feat_candidates = [
    # slice-level label
    "any_pos_rate",
    "total_area_sum",
    "total_frac_mean",
    # per-class label
    "label_pos_rate_large_bowel",
    "label_pos_rate_small_bowel",
    "label_pos_rate_stomach",
    "label_pos_mean_area_large_bowel",
    "label_pos_mean_area_small_bowel",
    "label_pos_mean_area_stomach",
    # intensity
    "zero_ratio",
    "mean",
    "std",
    "p01",
    "p50",
    "p99",
    "entropy",
    # cluster confidence
    "max_prob",
    "second_prob",
]
feature_cols = [c for c in feat_candidates if c in merged.columns]

corrs = corr_table(merged, target_cols, feature_cols)
OUT_CORR = OUT_DIR / "pearson_correlations.csv"
corrs.to_csv(OUT_CORR, index=False)
print("wrote:", OUT_CORR)

corrs.groupby("target").head(8)

## Plots

Write plots to the run directory `model_error_analysis/`.

In [ ]:
# dice_mean histogram
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.hist(merged["dice_mean"].to_numpy(dtype=float), bins=30)
ax.set_title("dice_mean distribution (per case_day)")
ax.set_xlabel("dice_mean")
ax.set_ylabel("count")
p = OUT_DIR / "dice_mean_hist.png"
fig.savefig(p, dpi=200)
print("wrote:", p)
plt.show()


In [ ]:
# scatter: dice_mean vs any_pos_rate
if "any_pos_rate" in merged.columns:
    fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
    ax.scatter(merged["any_pos_rate"], merged["dice_mean"], s=12, alpha=0.7)
    ax.set_xlabel("any_pos_rate (slice-level)")
    ax.set_ylabel("dice_mean")
    ax.set_title("dice_mean vs any_pos_rate")
    p = OUT_DIR / "scatter_dice_mean_vs_any_pos_rate.png"
    fig.savefig(p, dpi=200)
    print("wrote:", p)
    plt.show()


In [ ]:
# boxplot: dice_mean by cluster
if "cluster_id" in merged.columns:
    order = merged.groupby("cluster_id")["dice_mean"].mean().sort_values(ascending=True).index.tolist()
    data = [merged.loc[merged["cluster_id"] == cid, "dice_mean"].to_numpy(dtype=float) for cid in order]
    fig, ax = plt.subplots(figsize=(max(8, len(order) * 0.5), 4), constrained_layout=True)
    ax.boxplot(data, tick_labels=[str(int(c)) for c in order], showfliers=False)
    ax.set_xlabel("cluster_id (sorted by mean dice)")
    ax.set_ylabel("dice_mean")
    ax.set_title("dice_mean by style cluster (GMM)")
    p = OUT_DIR / "boxplot_dice_mean_by_cluster.png"
    fig.savefig(p, dpi=200)
    print("wrote:", p)
    plt.show()


## Cluster summary

Write `dice_by_cluster.csv` under the run directory.

In [ ]:
if "cluster_id" in merged.columns:
    cluster_summary = (
        merged.groupby("cluster_id", as_index=False)
        .agg(
            n=("case_day", "size"),
            dice_mean_mean=("dice_mean", "mean"),
            dice_mean_median=("dice_mean", "median"),
            dice_mean_p10=("dice_mean", lambda x: float(np.quantile(x, 0.10))),
        )
        .sort_values("dice_mean_mean", ascending=True)
    )
    OUT_CLUSTER = OUT_DIR / "dice_by_cluster.csv"
    cluster_summary.to_csv(OUT_CLUSTER, index=False)
    print("wrote:", OUT_CLUSTER)
    cluster_summary

## Inspect `cluster_id == 1`

Goal: understand what cluster 1 looks like in raw scans, and how it differs from other clusters.

We do two things:
1. Compare summary statistics (intensity + label occupancy) between cluster 1 and the rest.
2. Visualize a few scan slices from selected `case_day`s in cluster 1 vs a comparison cluster.


In [ ]:
assert "cluster_id" in merged.columns, "cluster_id not found; generate clustering first."

cluster1 = merged.loc[merged["cluster_id"] == 1].copy()
others = merged.loc[merged["cluster_id"] != 1].copy()

print("cluster1 n:", len(cluster1))
print("others   n:", len(others))

display(
    pd.DataFrame(
        {
            "group": ["cluster_id==1", "others"],
            "dice_mean_mean": [cluster1["dice_mean"].mean(), others["dice_mean"].mean()],
            "dice_mean_median": [cluster1["dice_mean"].median(), others["dice_mean"].median()],
            "any_pos_rate_mean": [cluster1.get("any_pos_rate", pd.Series([np.nan])).mean(), others.get("any_pos_rate", pd.Series([np.nan])).mean()],
            "total_frac_mean_mean": [cluster1.get("total_frac_mean", pd.Series([np.nan])).mean(), others.get("total_frac_mean", pd.Series([np.nan])).mean()],
            "zero_ratio_mean": [cluster1.get("zero_ratio", pd.Series([np.nan])).mean(), others.get("zero_ratio", pd.Series([np.nan])).mean()],
            "mean_intensity_mean": [cluster1.get("mean", pd.Series([np.nan])).mean(), others.get("mean", pd.Series([np.nan])).mean()],
            "std_intensity_mean": [cluster1.get("std", pd.Series([np.nan])).mean(), others.get("std", pd.Series([np.nan])).mean()],
            "entropy_mean": [cluster1.get("entropy", pd.Series([np.nan])).mean(), others.get("entropy", pd.Series([np.nan])).mean()],
            "p99_mean": [cluster1.get("p99", pd.Series([np.nan])).mean(), others.get("p99", pd.Series([np.nan])).mean()],
        }
    )
)

# Quick boxplots: cluster 1 vs others
feat_to_plot = [c for c in ["zero_ratio", "mean", "std", "entropy", "p99", "any_pos_rate", "total_frac_mean"] if c in merged.columns]

fig, axes = plt.subplots(1, len(feat_to_plot), figsize=(max(10, 2.2 * len(feat_to_plot)), 4), constrained_layout=True)
if len(feat_to_plot) == 1:
    axes = [axes]

for ax, col in zip(axes, feat_to_plot):
    data = [cluster1[col].to_numpy(dtype=float), others[col].to_numpy(dtype=float)]
    ax.boxplot(data, tick_labels=["cluster1", "others"], showfliers=False)
    ax.set_title(col)

p = OUT_DIR / "boxplot_cluster1_vs_others_features.png"
fig.savefig(p, dpi=200)
print("wrote:", p)
plt.show()

In [ ]:
try:
    import cv2
except Exception as e:
    raise RuntimeError("OpenCV (cv2) is required for visualization.") from e


def scan_dir_for_case_day(case_day: str) -> Path:
    # case113_day16 -> inputs/train/case113/case113_day16/scans
    case = case_day.split("_day", 1)[0]
    return Path("inputs/train") / case / case_day / "scans"


def list_scan_paths(case_day: str) -> list[Path]:
    d = scan_dir_for_case_day(case_day)
    if not d.is_dir():
        return []
    return sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() == ".png"])


def read_uint16_grayscale(p: Path) -> np.ndarray:
    img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise RuntimeError(f"Failed to read: {p}")
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img


def show_case_day_samples(
    case_day: str,
    *,
    n: int = 6,
    seed: int = 0,
    out_dir: Path | None = None,
    tag: str = "",
):
    paths = list_scan_paths(case_day)
    if not paths:
        print(f"No scans found for {case_day}")
        return
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(paths), size=min(n, len(paths)), replace=False)
    chosen = [paths[int(i)] for i in sorted(idx.tolist())]

    imgs = [read_uint16_grayscale(p) for p in chosen]
    # Use robust vmin/vmax from sampled non-zero pixels to improve visibility
    flat = np.concatenate([im.reshape(-1) for im in imgs], axis=0)
    nz = flat[flat != 0]
    if nz.size:
        vmin = float(np.quantile(nz, 0.01))
        vmax = float(np.quantile(nz, 0.99))
    else:
        vmin, vmax = 0.0, float(flat.max() if flat.size else 1.0)

    cols = min(3, len(imgs))
    rows = int(np.ceil(len(imgs) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), constrained_layout=True)
    axes = np.array(axes).reshape(-1)
    for ax, im, p in zip(axes, imgs, chosen):
        ax.imshow(im, cmap="gray", vmin=vmin, vmax=vmax)
        ax.set_title(p.name)
        ax.axis("off")
    for ax in axes[len(imgs) :]:
        ax.axis("off")
    fig.suptitle(case_day)
    if out_dir is not None:
        out_dir.mkdir(parents=True, exist_ok=True)
        stem = f"{tag}_{case_day}" if tag else case_day
        grid_path = out_dir / f"vis_{stem}_grid.png"
        fig.savefig(grid_path, dpi=200)
        print("wrote:", grid_path)
    plt.show()
    plt.close(fig)

    # Histogram (sampled pixels)
    samp = nz
    if samp.size > 500_000:
        samp = rng.choice(samp, size=500_000, replace=False)
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.hist(samp.astype(np.float64), bins=100)
    ax.set_title(f"Non-zero intensity histogram (sampled) - {case_day}")
    ax.set_xlabel("intensity")
    ax.set_ylabel("count")
    if out_dir is not None:
        stem = f"{tag}_{case_day}" if tag else case_day
        hist_path = out_dir / f"vis_{stem}_hist.png"
        fig.savefig(hist_path, dpi=200)
        print("wrote:", hist_path)
    plt.show()
    plt.close(fig)


# Choose a comparison cluster: the best cluster by mean dice
best_cluster = merged.groupby("cluster_id")["dice_mean"].mean().idxmax()
print("best_cluster_id (by mean dice):", best_cluster)

# Pick representative case_days
c1_worst = cluster1.sort_values("dice_mean").iloc[0]["case_day"]
c1_med = cluster1.sort_values("dice_mean").iloc[len(cluster1) // 2]["case_day"]

c_best = merged.loc[merged["cluster_id"] == best_cluster].copy()
best_med = c_best.sort_values("dice_mean").iloc[len(c_best) // 2]["case_day"]

print("cluster1 worst:", c1_worst)
print("cluster1 median:", c1_med)
print("best cluster median:", best_med)

# Persist the selected case_days for traceability
selected = pd.DataFrame(
    [
        {"role": "cluster1_worst", "case_day": c1_worst},
        {"role": "cluster1_median", "case_day": c1_med},
        {"role": "best_cluster_median", "case_day": best_med},
    ]
)
selected = selected.merge(merged[["case_day", "cluster_id", "dice_mean"]], on="case_day", how="left")
OUT_SELECTED = OUT_DIR / "selected_case_days.csv"
selected.to_csv(OUT_SELECTED, index=False)
print("wrote:", OUT_SELECTED)
display(selected)

for cd in [c1_worst, c1_med, best_med]:
    display(merged.loc[merged["case_day"] == cd, ["case_day", "dice_mean", "dice_large_bowel", "dice_small_bowel", "dice_stomach", "zero_ratio", "mean", "std", "entropy", "p99"]])

# Visualize a few slices
show_case_day_samples(c1_worst, n=6, seed=0, out_dir=OUT_DIR, tag="cluster1_worst")
show_case_day_samples(c1_med, n=6, seed=1, out_dir=OUT_DIR, tag="cluster1_median")
show_case_day_samples(best_med, n=6, seed=2, out_dir=OUT_DIR, tag="best_cluster_median")

In [ ]:
# Save a cluster-1 vs others feature comparison table for offline analysis
feat_cols = [c for c in ["zero_ratio", "mean", "std", "entropy", "p01", "p50", "p99", "any_pos_rate", "total_frac_mean"] if c in merged.columns]
rows = []
for c in feat_cols:
    c1 = cluster1[c].to_numpy(dtype=float)
    oth = others[c].to_numpy(dtype=float)
    rows.append(
        {
            "feature": c,
            "cluster1_mean": float(np.nanmean(c1)),
            "cluster1_median": float(np.nanmedian(c1)),
            "others_mean": float(np.nanmean(oth)),
            "others_median": float(np.nanmedian(oth)),
            "mean_diff": float(np.nanmean(c1) - np.nanmean(oth)),
            "median_diff": float(np.nanmedian(c1) - np.nanmedian(oth)),
        }
    )

cmp = pd.DataFrame.from_records(rows).sort_values("feature")
OUT_CMP = OUT_DIR / "cluster1_vs_others_feature_summary.csv"
cmp.to_csv(OUT_CMP, index=False)
print("wrote:", OUT_CMP)
cmp